### Getting started with pyspi

pyspi computes **statistics of pairwise interactions** (SPIs) between the processes of a
multivariate time series. Each SPI is one way of answering "how are these two processes
related?" -- a correlation coefficient, a spectral coherence, a transfer entropy, a
causal-inference score. For a dataset of `M` processes, every SPI returns an `M x M` matrix
of pairwise values, and pyspi stacks all of those matrices into a single table so that
hundreds of different notions of "related" can be compared side by side.

> Plotting cells need matplotlib, which is not a core pyspi dependency:
> `pip install matplotlib`, or `pip install "pyspi[bench]"`.

In [1]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspi.calculator import Calculator, bundled_configs
from pyspi.data import load_dataset, available_datasets

pd.set_option("display.width", 120)
available_datasets()

{'forex': 'Foreign-exchange rates (250 obs, 7 processes).',
 'cml': 'Coupled map lattice (500 obs, 10 processes).',
 'standard_normal': 'i.i.d. standard normal null model (200 obs, 5 processes).'}

#### 1. Load a dataset

`load_dataset` returns a `Data` object: an `M x T` array of `M` processes by `T`
observations, z-scored per process by default. We use `forex`, a real dataset of
foreign-exchange rates.

Your own data goes in the same way: `Calculator(dataset=my_array)` accepts a NumPy array,
and `dim_order` on `Data` controls whether rows are processes or observations.

In [2]:
data = load_dataset("forex")
print(f"{data.n_processes} processes x {data.n_observations} observations")
print("process names:", data.procnames)

7 processes x 250 observations
process names: ['proc-0', 'proc-1', 'proc-2', 'proc-3', 'proc-4', 'proc-5', 'proc-6']


#### 2. Run a calculator

`config` chooses which SPIs to compute. `fabfour` is the smallest bundled config: four
SPIs, one from each of four different families.

In [6]:
calc = Calculator(dataset=data, 
                  config="benchmarked_p90", 
                  verbose=True,
                  )

calc.compute(progress=True)
calc

# t0 = time.perf_counter()
# calc.compute(progress=False)
# fabfour_seconds = time.perf_counter() - t0

# print(f"{calc.n_spis} SPIs in {fabfour_seconds:.2f}s")
# list(calc.spis)

Loading configuration file: /Users/wedi0306/Code/pyspi-fork/pyspi/configs/benchmarked_p90.yaml
Importing module .statistics.basic
[1] .statistics.basic.Covariance(x,y,{'estimator': 'EmpiricalCovariance'}) -> "cov_EmpiricalCovariance"
[2] .statistics.basic.Covariance(x,y,{'estimator': 'EllipticEnvelope'}) -> "cov_EllipticEnvelope"
[3] .statistics.basic.Covariance(x,y,{'estimator': 'GraphicalLasso'}) -> "cov_GraphicalLasso"
[4] .statistics.basic.Covariance(x,y,{'estimator': 'GraphicalLassoCV'}) -> "cov_GraphicalLassoCV"
[5] .statistics.basic.Covariance(x,y,{'estimator': 'LedoitWolf'}) -> "cov_LedoitWolf"
[6] .statistics.basic.Covariance(x,y,{'estimator': 'MinCovDet'}) -> "cov_MinCovDet"
[7] .statistics.basic.Covariance(x,y,{'estimator': 'OAS'}) -> "cov_OAS"
[8] .statistics.basic.Covariance(x,y,{'estimator': 'ShrunkCovariance'}) -> "cov_ShrunkCovariance"
[9] .statistics.basic.Covariance(x,y,{'estimator': 'EmpiricalCovariance', 'squared': True}) -> "cov-sq_EmpiricalCovariance"
[10] .statis


SPI Computation Results Summary

Total number of SPIs attempted: 293
Number of SPIs successfully computed: 285 (97.27%)
------------------------------------------------------------
Total compute time: 5.01s across 293 timed SPI(s)
Slowest 5:
  ddtf_multitaper_mean_fs-1_fmin-0_fmax-0-5       0.58s  ( 11.6%)
  gpdcoh_multitaper_mean_fs-1_fmin-0_fmax-0-5     0.43s  (  8.7%)
  dcoh_multitaper_mean_fs-1_fmin-0_fmax-0-5       0.43s  (  8.6%)
  dtf_multitaper_mean_fs-1_fmin-0_fmax-0-5        0.37s  (  7.4%)
  pdcoh_multitaper_mean_fs-1_fmin-0_fmax-0-5      0.31s  (  6.2%)
------------------------------------------------------------
Category       | Count | Percentage
------------------------------------------------------------
Successful     |   285 |  97.27%
NaNs           |     8 |   2.73%
Partial NaNs   |     0 |   0.00%
------------------------------------------------------------

[8] SPI(s) produced NaN outputs:
------------------------------------------------------------
1. je_kozachen

#### 3. Reading the results table

`calc.table` is a `DataFrame` whose rows are processes and whose **columns are a two-level
`MultiIndex` of `(spi, process)`**. Its shape is therefore `M x (n_spis * M)`: every SPI's
`M x M` matrix, laid side by side. Each matrix has a `NaN` diagonal -- self-pairs are not
computed.

**Orientation matters.** Entry `[i, j]` is computed with process `i` as the *source* and
process `j` as the *target*. For an undirected SPI the matrix is symmetric and the
distinction is irrelevant; for a directed SPI it is the whole point.

In [8]:
print("table shape:", calc.table.shape, "| column levels:", calc.table.columns.names)
calc.table.iloc[:3, :6]

table shape: (7, 2051) | column levels: ['spi', 'process']


spi     cov_EmpiricalCovariance                                                  
process                  proc-0    proc-1    proc-2    proc-3    proc-4    proc-5
proc-0                      NaN -0.676742 -0.171030 -0.649407  0.382806 -0.539182
proc-1                -0.676742       NaN  0.150468  0.901080 -0.107627  0.416836
proc-2                -0.171030  0.150468       NaN  0.090539 -0.195452  0.141439

##### Slicing the table

Indexing by an SPI identifier gives that SPI's `M x M` matrix back as a `DataFrame`.
Indexing the full table with the `(spi, process)` pair gives a single value.

In [7]:
cov = calc.table["cov_EmpiricalCovariance"]
display(cov.round(3))

# The data is z-scored, so this SPI is the Pearson correlation.
print("proc-1 / proc-3:", calc.table.loc["proc-1", ("cov_EmpiricalCovariance", "proc-3")].round(4))

# Strongest pairs, upper triangle only (the lower one is redundant here).
upper = cov.where(np.triu(np.ones(cov.shape), k=1).astype(bool))
upper.stack().abs().sort_values(ascending=False).head(5).round(3)

process,proc-0,proc-1,proc-2,proc-3,proc-4,proc-5,proc-6
proc-0,NaN,-0.677,-0.171,-0.649,0.383,-0.539,0.482
proc-1,-0.677,NaN,0.150,0.901,-0.108,0.417,-0.859
proc-2,-0.171,0.150,NaN,0.091,-0.195,0.141,-0.016
proc-3,-0.649,0.901,0.091,NaN,-0.060,0.353,-0.816
proc-4,0.383,-0.108,-0.195,-0.060,NaN,-0.657,-0.271
proc-5,-0.539,0.417,0.141,0.353,-0.657,NaN,-0.204
proc-6,0.482,-0.859,-0.016,-0.816,-0.271,-0.204,NaN


proc-1 / proc-3: 0.9011


        process
proc-1  proc-3     0.901
        proc-6     0.859
proc-3  proc-6     0.816
proc-0  proc-1     0.677
proc-4  proc-5     0.657
dtype: float64

##### Directed vs undirected

`di_gaussian` (directed information) is directed, so its matrix is *not* symmetric. The
two triangles are separate estimates and their difference is meaningful.

In [ ]:
di = calc.table["di_gaussian"]
print("proc-0 -> proc-1:", di.loc["proc-0", "proc-1"].round(3),
      " | proc-1 -> proc-0:", di.loc["proc-1", "proc-0"].round(3))
print("di symmetric?", np.allclose(di.values, di.values.T, equal_nan=True),
      "| cov symmetric?", np.allclose(cov.values, cov.values.T, equal_nan=True))
di.round(3)

#### 4. Configs, and why they matter

There is no free lunch: the full config is ~328 SPIs and some of them are expensive.
`calc.timings` reports measured wall-clock seconds per SPI from the last `compute()`,
which is the honest way to budget a run.

Below we rerun the same data under `sonnet` (14 SPIs, one representative per family).
This is the slow cell in the notebook -- expect tens of seconds. Several SPIs warn on
this dataset because 250 observations is short for spectral and wavelet estimators; they
still return values.

In [ ]:
print("bundled configs:", bundled_configs())

calc_sonnet = Calculator(dataset=data, config="sonnet", verbose=False)

t0 = time.perf_counter()
calc_sonnet.compute(progress=False)
sonnet_seconds = time.perf_counter() - t0

print(f"\nfabfour: {calc.n_spis:2d} SPIs in {fabfour_seconds:6.2f}s")
print(f"sonnet : {calc_sonnet.n_spis:2d} SPIs in {sonnet_seconds:6.2f}s")

pd.Series(calc_sonnet.timings).sort_values(ascending=False).round(3).to_frame("seconds")

The cost is heavily skewed: a handful of SPIs dominate the total while most are
effectively free. That skew is what the `benchmarked_p80` / `p90` / `p95` / `p99` configs
exploit -- they keep the fastest N% of SPIs by measured amortized cost, so
`benchmarked_p80` is the cheapest and `benchmarked_p99` the most complete.

To build your own subset by keyword, `pyspi.utils.filter_spis` writes a filtered config
from the SPI labels, e.g. `filter_spis(["directed", "linear"], output_name="my_config")`,
which you then pass as `Calculator(config="my_config.yaml")`.

#### 5. Visualising one SPI

A single SPI matrix is a small heatmap. Covariance is signed, so it gets a diverging
colour map centred on zero; an unsigned SPI (a distance, an information rate) should use
a single-hue sequential map instead.

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.4))

vmax = np.nanmax(np.abs(cov.values))
im = ax.imshow(cov.values, cmap="coolwarm", vmin=-vmax, vmax=vmax)

ax.set_xticks(range(data.n_processes), data.procnames, rotation=45, ha="right")
ax.set_yticks(range(data.n_processes), data.procnames)
ax.set_title("Covariance (z-scored $\\Rightarrow$ Pearson correlation)", fontsize=11)
ax.set_xlabel("target")
ax.set_ylabel("source")

for i in range(data.n_processes):
    for j in range(data.n_processes):
        if i != j:
            v = cov.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(v) > 0.6 * vmax else "#333333")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

#### 6. Parallelism on a single dataset

`compute(n_jobs=N)` spreads the SPIs over `N` worker **processes** -- not threads. The
default is `1` (serial); `PYSPI_N_JOBS` sets it globally if you would rather not pass it
at every call site.

The cell below runs the same `sonnet` config on one synthetic dataset at four values of
`n_jobs` and times each with `time.perf_counter()`; it takes a minute or so. The series
is longer than `forex` on purpose. Worker startup is not free -- a few seconds under the
`spawn` start method used on macOS and Windows -- so on a run of only a few seconds that
overhead swamps whatever parallelism buys you.

In [ ]:
X = np.random.default_rng(0).standard_normal((6, 400))

runs = []
for n_jobs in (1, 2, 4, 8):
    c = Calculator(dataset=X, config="sonnet", verbose=False)
    t0 = time.perf_counter()
    c.compute(n_jobs=n_jobs, progress=False)
    runs.append((n_jobs, time.perf_counter() - t0))

serial = runs[0][1]
longest_spi = max(c.timings.values())  # per-SPI timings are recorded in parallel runs too

print(f"{os.cpu_count()} logical cores | longest single SPI {longest_spi:.1f}s "
      f"=> best possible speedup <= {serial / longest_spi:.1f}x")
pd.DataFrame([(n, s, serial / s) for n, s in runs],
             columns=["n_jobs", "seconds", "speedup"]).set_index("n_jobs").round(2)

The speedup plateaus well short of `n_jobs`, then typically turns back down once
`n_jobs` approaches or exceeds the core count. That is structural rather than a tuning
problem: SPIs that share a cache -- a spectral estimate, a state-space embedding -- are
grouped into a single task that runs sequentially inside one worker, so the total is
floored by the longest task. The bound printed above measures that floor with the single
longest SPI, so it is if anything optimistic -- the real floor is the longest *group* --
and no value of `n_jobs` beats it. Measured across the benchmark grid at `M=64, T=1600`
that ceiling sits at 2.3-4.5x on every bundled config.

**The practical consequence: with many datasets, run one dataset per process** -- a
cluster array job, or GNU parallel -- rather than raising `n_jobs`. That scales close to
linearly and confines a failure to one dataset. Raise `n_jobs` only when you have fewer
datasets than cores, or when one dataset has to finish inside a walltime limit.

One platform caveat: on macOS the Accelerate BLAS cannot be thread-pinned, so
`n_jobs > 1` can oversubscribe cores on BLAS-heavy SPIs and the plateau arrives sooner.
Elsewhere pyspi pins each worker's nested thread pools to a single thread.

#### 7. The same run from the command line

Everything above is available without writing Python. Save your data as an `.npy` array
and run:

```bash
python -m pyspi compute --data ts.npy --config fabfour --n-jobs 4 --checkpoint-dir results/
```

`--checkpoint-dir` writes each finished SPI to disk as it completes, so an interrupted
run resumes instead of restarting. The same applies in Python:
`calc.compute(checkpoint_dir="results/", resume=True)`.

##### Next

`02_comparing_spis.ipynb` builds a dataset with known couplings and shows where the
choice of SPI actually changes the answer.